# Chatbot with memory

**dabeer.ai · AI for Beginners · Python and AI APIs**

In this notebook you'll build a small chatbot that remembers the conversation, watch the token count grow with every turn, and try a system prompt and two temperatures.

**How to use it:** run the cells from top to bottom. Click the play button next to a cell, or press Shift+Enter. Read the comments (lines starting with `#`), then change one value at a time and run again.

**Before you start:** add your API key to Colab Secrets (the key icon in the left sidebar), named `GEMINI_API_KEY`, `ANTHROPIC_API_KEY` or `OPENAI_API_KEY`, with **Notebook access** switched on. The lab guide shows how. Never type your key into a cell.

### No paid API key? Use Gemini's free key

This notebook starts with `PROVIDER = "gemini"`, because Google AI Studio gives you a free Gemini API key, with no payment method needed.

1. Go to https://aistudio.google.com/apikey and sign in with your Google account.
2. Click **Create API key**, and copy the key.
3. In Colab, click the **key icon** in the left sidebar, add a secret named `GEMINI_API_KEY`, paste the key as the value, and switch on **Notebook access**.

Two things to know about the free key:

- It has rate limits. If you see a "429" or "resource exhausted" error, wait a minute and run the cell again.
- On the free tier, Google may use what you send, and the replies, to improve its products, and people may read them. So use only fictional or practice data, like the samples here. Never real customer, company or personal data.

Have a paid Anthropic or OpenAI key instead? Set `PROVIDER` to `"anthropic"` or `"openai"` in the settings cell.

## Step 1: Install the libraries

One line installs all three providers' Python libraries. You only need one of them, but installing all three lets you switch later.

In [ ]:
!pip install -q google-genai anthropic openai

## Step 2: Settings

This is the **only** cell you need to edit to switch provider or model.

In [ ]:
# ---- Settings: change these ----

PROVIDER = "gemini"   # "gemini" (free key), "anthropic" or "openai"

# Model names change. Check the provider's model list for current names.
MODELS = {
    "gemini": "gemini-3.1-flash-lite",   # model names change: pick a current one from the AI Studio list
    "anthropic": "claude-sonnet-5-5",
    "openai": "gpt-5-mini",
}
MODEL = MODELS[PROVIDER]

MAX_TOKENS = 2000   # the most tokens one reply may use

# The system prompt: rules that apply to every reply.
SYSTEM_PROMPT = """You are a friendly assistant for Sweet Crumbs, a small bakery in Pune.
Help customers choose cakes and place orders.
Keep every answer short: three sentences at most."""

TEMPERATURE = None   # None means the model's default. You'll try 0 and 1 later.

## Step 3: Connect to the provider

This reads your key from Colab Secrets and creates a **client**, your phone line to the provider. If you see an error here, check the secret's name and that **Notebook access** is on.

In [ ]:
# Reads your API key from Colab Secrets, so it never appears in this notebook.
from google.colab import userdata

KEY_NAMES = {"gemini": "GEMINI_API_KEY", "anthropic": "ANTHROPIC_API_KEY", "openai": "OPENAI_API_KEY"}

try:
    api_key = userdata.get(KEY_NAMES[PROVIDER])
except Exception as error:
    raise RuntimeError(
        f"Couldn't read the secret {KEY_NAMES[PROVIDER]}. Open the key icon in the left sidebar, "
        "check the name is exactly right, and switch on 'Notebook access'. "
        f"Details: {error}"
    ) from None

if PROVIDER == "gemini":
    from google import genai
    from google.genai import types
    client = genai.Client(api_key=api_key)
elif PROVIDER == "anthropic":
    import anthropic
    client = anthropic.Anthropic(api_key=api_key)
elif PROVIDER == "openai":
    import openai
    client = openai.OpenAI(api_key=api_key)
else:
    raise ValueError('PROVIDER must be "gemini", "anthropic" or "openai"')

print("Ready. Provider:", PROVIDER, "| Model:", MODEL)

## Step 4: One function that talks to the model

`send_to_model` sends a list of messages and returns three things: the reply text, the tokens that went in, and the tokens that came out. The three providers put the system prompt and the reply text in slightly different places, so there's one branch for each.

In [ ]:
def send_to_model(messages, temperature=None):
    """Send a list of {"role": ..., "content": ...} messages.
    Returns (reply_text, tokens_in, tokens_out)."""
    if temperature is None:
        temperature = TEMPERATURE
    extra = {}
    if temperature is not None:
        extra["temperature"] = temperature

    if PROVIDER == "gemini":
        # Gemini calls the assistant "model", and puts each message's text inside "parts"
        contents = [{"role": "model" if m["role"] == "assistant" else "user",
                     "parts": [{"text": m["content"]}]} for m in messages]
        response = client.models.generate_content(
            model=MODEL,
            contents=contents,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_PROMPT,   # Gemini: the system prompt goes in the config
                max_output_tokens=MAX_TOKENS,
                **extra,
            ),
        )
        reply = response.text or ""
        if response.candidates and response.candidates[0].finish_reason == "MAX_TOKENS":
            reply += " [reply cut off: raise MAX_TOKENS]"
        usage = response.usage_metadata
        # The model's hidden "thinking" tokens are billed as output too, so we count them
        tokens_out = (usage.candidates_token_count or 0) + (usage.thoughts_token_count or 0)
        return reply, usage.prompt_token_count, tokens_out

    elif PROVIDER == "anthropic":
        response = client.messages.create(
            model=MODEL,
            max_tokens=MAX_TOKENS,
            system=SYSTEM_PROMPT,          # Anthropic: the system prompt is its own setting
            messages=messages,
            **extra,
        )
        reply = "".join(block.text for block in response.content if block.type == "text")
        if response.stop_reason == "max_tokens":
            reply += " [reply cut off: raise MAX_TOKENS]"
        return reply, response.usage.input_tokens, response.usage.output_tokens

    else:
        response = client.chat.completions.create(
            model=MODEL,
            max_completion_tokens=MAX_TOKENS,
            # OpenAI: the system prompt goes in as the first message
            messages=[{"role": "system", "content": SYSTEM_PROMPT}] + messages,
            **extra,
        )
        reply = response.choices[0].message.content or ""
        if response.choices[0].finish_reason == "length":
            reply += " [reply cut off: raise MAX_TOKENS]"
        return reply, response.usage.prompt_tokens, response.usage.completion_tokens


# A quick test: one message, no memory yet.
reply, tokens_in, tokens_out = send_to_model([{"role": "user", "content": "Say hello in one short sentence."}])
print(reply)
print(f"(tokens in: {tokens_in}, tokens out: {tokens_out})")

## Step 5: Add memory

A model doesn't remember anything between calls. The "memory" is just a Python **list** called `history`. Every turn, we add your message, send the **whole list**, and then add the reply. `token_log` records the tokens of each turn so you can watch them grow.

In [ ]:
history = []     # the whole conversation, as a list of {"role": ..., "content": ...}
token_log = []   # one row per turn

def chat(user_message, show_question=True):
    history.append({"role": "user", "content": user_message})
    try:
        reply, tokens_in, tokens_out = send_to_model(history)
    except Exception:
        history.pop()   # remove the message that failed, so the history stays clean
        raise
    history.append({"role": "assistant", "content": reply})
    token_log.append({"turn": len(token_log) + 1, "you said": user_message[:40],
                      "tokens in": tokens_in, "tokens out": tokens_out})
    if show_question:
        print("You:", user_message)
    print("Bot:", reply)
    print(f"     (tokens in: {tokens_in}, tokens out: {tokens_out})\n")

def reset_chat():
    history.clear()
    token_log.clear()
    print("Conversation cleared.")

## Step 6: Talk to it

Run this cell. The third question only works if the bot remembers the first two.

In [ ]:
reset_chat()
chat("Hi, I'm Asha. I'd like a cake for my daughter's seventh birthday on Saturday.")
chat("She loves mango. What would you suggest?")
chat("What's my name, and what's the cake for?")

## Step 7: Now without memory

This sends **only** the last question, with no history. Compare the answer with the one above.

In [ ]:
reply, tokens_in, tokens_out = send_to_model([{"role": "user", "content": "What's my name, and what's the cake for?"}])
print("Bot without memory:", reply)
print(f"     (tokens in: {tokens_in}, tokens out: {tokens_out})")

## Step 8: Chat in a loop

Type a message and press Enter. Type `quit` to stop. Have at least five more turns, then stop and run the next cell.

In [ ]:
print('Type a message and press Enter. Type "quit" to stop.')
while True:
    user_message = input("You: ").strip()
    if user_message.lower() in ("quit", "exit", "bye"):
        break
    if user_message:
        chat(user_message, show_question=False)

## Step 9: Watch the tokens grow

Every turn sends the whole conversation again, so **tokens in** keeps climbing even when your messages are short.

In [ ]:
import pandas as pd
from IPython.display import display

table = pd.DataFrame(token_log)
display(table)
table.plot(x="turn", y=["tokens in", "tokens out"], kind="bar", title="Tokens per turn")

## Step 10: Estimate the cost

Open your provider's pricing page, find your model, and type its prices below, **per 1 million tokens**, in your account's currency. This notebook doesn't include prices, because they change.

In [ ]:
PRICE_IN_PER_MILLION = 0.0    # price for 1 million input tokens
PRICE_OUT_PER_MILLION = 0.0   # price for 1 million output tokens

total_in = sum(row["tokens in"] for row in token_log)
total_out = sum(row["tokens out"] for row in token_log)
cost = total_in / 1_000_000 * PRICE_IN_PER_MILLION + total_out / 1_000_000 * PRICE_OUT_PER_MILLION

print(f"Turns: {len(token_log)}  |  tokens in: {total_in}  |  tokens out: {total_out}")
print(f"Estimated cost of this conversation: {cost:.4f}")
print(f"Estimated cost of 1,000 conversations like it: {cost * 1000:.2f}")

## Step 11: Try two temperatures

The same request, three times each, at a low and a high temperature. Low should give similar names every time; high, more varied ones.

Some models, especially reasoning models, don't accept a temperature setting. If you see the "did not accept" message, that's expected: it's the model telling you, not a bug in your code. You can switch `MODEL` in the settings cell to one from the provider's list that accepts temperature, or just note the result and move on.

In [ ]:
def try_temperature(temperature, runs=3):
    prompt = "Suggest a name for a new mango cake. Reply with the name only."
    print(f"Temperature {temperature}:")
    for run in range(1, runs + 1):
        try:
            reply, _, _ = send_to_model([{"role": "user", "content": prompt}], temperature=temperature)
            print(f"  run {run}: {reply.strip()}")
        except Exception as error:
            if "429" in str(error) or "RESOURCE_EXHAUSTED" in str(error):   # free-key rate limit, not temperature
                print("  Rate limit reached. Wait a minute, then run this cell again.")
            else:
                print(f"  This model did not accept temperature={temperature}. Details: {error}")
            return

try_temperature(0)
try_temperature(1)

## Step 12: Change the system prompt

Edit `SYSTEM_PROMPT` in the settings cell (for example, make the bot reply in Hindi, or only talk about cakes and politely refuse anything else). Run the settings cell, then run the cell below. You don't need to reconnect.

In [ ]:
reset_chat()
chat("Hello, what do you sell?")
chat("Can you help me with my maths homework?")

## Stretch: a shorter memory

Long chats get expensive because every turn resends everything. One fix: send only the last few messages. `KEEP_LAST` must be an odd number, so the list starts with one of your messages.

In [ ]:
KEEP_LAST = 5   # your latest message plus the last two exchanges

def chat_short_memory(user_message):
    history.append({"role": "user", "content": user_message})
    recent = history[-KEEP_LAST:]
    try:
        reply, tokens_in, tokens_out = send_to_model(recent)
    except Exception:
        history.pop()
        raise
    history.append({"role": "assistant", "content": reply})
    token_log.append({"turn": len(token_log) + 1, "you said": user_message[:40],
                      "tokens in": tokens_in, "tokens out": tokens_out})
    print("You:", user_message)
    print("Bot:", reply)
    print(f"     (tokens in: {tokens_in}, tokens out: {tokens_out})\n")

reset_chat()
chat_short_memory("Hi, I'm Asha. I'd like a cake for my daughter's seventh birthday.")
chat_short_memory("She loves mango.")
chat_short_memory("Can it be eggless?")
chat_short_memory("Can you deliver to Kothrud?")
chat_short_memory("What's my name?")